

### Train a conditional VAE (CVAE) over MINIST


(a) Define the DNN architecture for CVAE

In [21]:


import torch
import torchvision
import numpy as np
import matplotlib.pyplot as plt
import torch.nn as nn
import torch.nn. functional as F
import torch.optim as optim
import os
import sys
from PIL import Image


batch_size = 100
learning_rate = 1e-3
max_epoch = 100
device = "cuda"
num_workers = 5
load_epoch = -1
generate = True

class Model(nn.Module):
    def __init__(self,latent_size=32,num_classes=10):
        super(Model,self).__init__()
        self.latent_size = latent_size
        self.num_classes = num_classes

        # For encode
        self.conv1 = nn.Conv2d(2, 16, kernel_size=5, stride=2)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=5, stride=2)
        self.linear1 = nn.Linear(4*4*32,300)
        self.mu = nn.Linear(300, self.latent_size)
        self.logvar = nn.Linear(300, self.latent_size)

        # For decoder
        self.linear2 = nn.Linear(self.latent_size + self.num_classes, 300)
        self.linear3 = nn.Linear(300,4*4*32)
        self.conv3 = nn.ConvTranspose2d(32, 16, kernel_size=5,stride=2)
        self.conv4 = nn.ConvTranspose2d(16, 1, kernel_size=5, stride=2)
        self.conv5 = nn.ConvTranspose2d(1, 1, kernel_size=4)

    def encoder(self,x,y):
        y = torch.argmax(y, dim=1).reshape((y.shape[0],1,1,1))
        y = torch.ones(x.shape).to(device)*y
        t = torch.cat((x,y),dim=1)

        t = F.relu(self.conv1(t))
        t = F.relu(self.conv2(t))
        t = t.reshape((x.shape[0], -1))

        t = F.relu(self.linear1(t))
        mu = self.mu(t)
        logvar = self.logvar(t)
        return mu, logvar

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5*logvar)
        eps = torch.randn_like(std).to(device)
        return eps*std + mu

    def unFlatten(self, x):
        return x.reshape((x.shape[0], 32, 4, 4))

    def decoder(self, z):
        t = F.relu(self.linear2(z))
        t = F.relu(self.linear3(t))
        t = self.unFlatten(t)
        t = F.relu(self.conv3(t))
        t = F.relu(self.conv4(t))
        t = F.relu(self.conv5(t))
        return t


    def forward(self, x, y):
        mu, logvar = self.encoder(x,y)
        z = self.reparameterize(mu,logvar)

        # Class conditioning
        z = torch.cat((z, y.float()), dim=1)
        pred = self.decoder(z)
        return pred, mu, logvar




(b) Create an instance named as "model" for the defined CVAE class and plot the architecture of the instance. Put the instance "model" to GPU.

(c) write code to convert a label y of numpy type to a onehot vector of dimension 10 of pytorch tensor. Put the onehot tensor to GPU

In [30]:
y = np.array(2)



tensor([[0., 0., 1., 0., 0., 0., 0., 0., 0., 0.]], device='cuda:0',
       dtype=torch.float64)


(c): Test the encoder function of the instance named "model" by creating a particular input x and computing the output of the encoder function with x and label (from (b)) as input.

Create x to be pytorch tensor of size (1,1,28,28) located at GPU device.

Print the output mu vector and logvar vector.

(d) Define the loss function for training the CVAE

In [36]:
def loss_function(x, pred, mu, logvar):

    return recon_loss, kld


(e) Complete the code for the training procedure

In [38]:


def train(epoch, model, train_loader, optim):
    reconstruction_loss = 0
    kld_loss = 0
    total_loss = 0
    for i,(x,y) in enumerate(train_loader):

        label = np.zeros((x.shape[0], 10))
        label[np.arange(x.shape[0]), y] = 1
        label = torch.tensor(label)


        # clean the gradient values to be zero in the optimizer


        # Compute the model output with x and label as input


        # Compute the loss and update the model parameters via backpropagation


        # Update reconstruction_loss, kld_loss, total_loss



    reconstruction_loss /= len(train_loader.dataset)
    kld_loss /= len(train_loader.dataset)
    total_loss /= len(train_loader.dataset)
    return total_loss, kld_loss,reconstruction_loss




(e)  Complete the code for the test procedure

In [39]:
def test(epoch, model, test_loader):
    reconstruction_loss = 0
    kld_loss = 0
    total_loss = 0
    with torch.no_grad():
        for i,(x,y) in enumerate(test_loader):

            label = np.zeros((x.shape[0], 10))
            label[np.arange(x.shape[0]), y] = 1
            label = torch.tensor(label)

            # Compute the loss with x and label as input to the model


            # Update reconstruction_loss, kld_loss, total_loss



    reconstruction_loss /= len(test_loader.dataset)
    kld_loss /= len(test_loader.dataset)
    total_loss /= len(test_loader.dataset)
    return total_loss, kld_loss,reconstruction_loss

(f) Start to train the CAVE model

In [ ]:
def generate_image(epoch,z, y, model):
    with torch.no_grad():
        label = np.zeros((y.shape[0], 10))
        label[np.arange(z.shape[0]), y] = 1
        label = torch.tensor(label)

        pred = model.decoder(torch.cat((z.to(device),label.float().to(device)), dim=1))


        #imgplot = plt.imshow(pred[0,0,:,:].to('cpu').numpy())
        tt = pred[0,0,:,:].to('cpu').numpy()

        tt = tt/(np.max(tt)+1e-10)*255
        plt.imshow(tt, cmap='gray', vmin=0, vmax=255)
        plt.show()


        plot(epoch, pred.cpu().data.numpy(), y.cpu().data.numpy(),name='Eval_')
        print("data Plotted")



def load_data():
    transform = torchvision.transforms.Compose([
                               torchvision.transforms.ToTensor()])
    train_loader = torch.utils.data.DataLoader(torchvision.datasets.MNIST('./data/', train=True, download=True,
                             transform=transform),batch_size=batch_size, num_workers=num_workers, shuffle=True)
    test_loader = torch.utils.data.DataLoader(torchvision.datasets.MNIST('./data/', train=False, download=True,
                             transform=transform),batch_size=batch_size, num_workers=num_workers, shuffle=True)

    return train_loader, test_loader


train_loader, test_loader = load_data()
print("dataloader created")
#model = Model().to(device)
#print("model created")


optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=0.001)


train_loss_list = []
test_loss_list = []
for i in range(load_epoch+1, max_epoch):
    model.train()
    train_total, train_kld, train_loss = train(i, model, train_loader, optimizer)
    with torch.no_grad():
        model.eval()
        test_total, test_kld, test_loss = test(i, model, test_loader)
        if generate:
            z = torch.randn(6, 32).to(device)
            y = torch.tensor([1,2,3,4,5,6]) - 1
            generate_image(i,z, y, model)


    print("Epoch: {}/{} Train loss: {}, Train KLD: {}, Train Reconstruction Loss:{}".format(i, max_epoch,train_total, train_kld, train_loss))
    print("Epoch: {}/{} Test loss: {}, Test KLD: {}, Test Reconstruction Loss:{}".format(i, max_epoch, test_loss, test_kld, test_loss))


    train_loss_list.append([train_total, train_kld, train_loss])
    test_loss_list.append([test_total, test_kld, test_loss])